# SymPy verifications of the main theorems from Section 4

The next cell setups SymPy.

In [1]:
import sympy as sp

def check(expr):
    return sp.simplify(expr)

## 1 Verification of Theorem 4.1

### 1.1 Statement and proof (as in the manuscript)

**Theorem 4.1. (Nesterov's method from unregularized models, $m>0$)** Let $f\in\mathcal{F}_{m,L}(\mathbb{R}^d)$ with $0<m<L$, $q=m/L$, let $k\geq1$, $A_k>0$, $a_k\geq0$, $A_{k+1}=A_k+a_k$, $\delta_k=\tfrac{a_k}{qA_{k+1}}$, let $A_kr_k=A_k\sum_{i<k}\lambda_i\ell_i$ for some $\lambda\in\Delta_k$ and $y_0,\ldots,y_{k-1}\in\mathbb{R}^d$ (a quadratic with curvature $mA_k$), with minimizer $z_k$, let $x_k,y_k\in\mathbb{R}^d$ satisfy the relation
$$
\langle\nabla f(y_k),\,A_k(x_k-y_k)+q\delta_kA_k\,(z_k-y_k)\rangle\geq0 ,
$$
let $x_{k+1}=y_k-\tfrac1L\nabla f(y_k)$, $A_{k+1}r_{k+1}=A_kr_k+a_k\ell_k$ with minimizer
$$
z_{k+1}=z_k+q\delta_k\,(y_k-z_k)-\tfrac{\delta_k}{L}\nabla f(y_k),
$$
and let $\Psi_k=A_k\big(f(x_k)-r_k(z_k)\big)$. If $qA_{k+1}^2-a_k^2\geq0$, then it holds that
$$
\Psi_{k+1}\leqslant\Psi_k .
$$

*Proof.* The proof consists in a weighted sum of the following inequalities:

- convexity between $x_k$ and $y_k$, with weight $\lambda_1=A_k$:

  $$
  f(x_k)\geq f(y_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle,
  $$

- the descent lemma, with weight $\lambda_2=A_{k+1}$:

  $$
  f(x_{k+1})\leq f(y_k)-\tfrac{1}{2L}\|\nabla f(y_k)\|^2,
  $$

- the relation, with weight $\lambda_3=1$:

  $$
  \langle\nabla f(y_k),\,A_k(x_k-y_k)+q\delta_kA_k\,(z_k-y_k)\rangle\geq0.
  $$

The weighted sum reads

$$
\begin{aligned}
0\geq{}&\lambda_1\big[f(y_k)-f(x_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle\big]\\
&+\lambda_2\big[f(x_{k+1})-f(y_k)+\tfrac{1}{2L}\|\nabla f(y_k)\|^2\big]\\
&+\lambda_3\langle\nabla f(y_k),\,A_k(y_k-x_k)+q\delta_kA_k\,(y_k-z_k)\rangle ,
\end{aligned}
$$

which, using the update of $z_{k+1}$ and $A_{k+1}=A_k+a_k$, together with $A_{k+1}r_{k+1}(z_{k+1})=A_kr_k(z_{k+1})+a_k\ell_k(z_{k+1})$ and $A_kr_k(z_{k+1})=A_kr_k(z_k)+\tfrac{mA_k}{2}\|z_{k+1}-z_k\|^2$ (as $A_kr_k$ is a quadratic with curvature $mA_k$ minimized at $z_k$), can be reformulated exactly as

$$
0\geq\Psi_{k+1}-\Psi_k+\tfrac{A_{k+1}-a_k\delta_k}{2L}\|\nabla f(y_k)\|^2+q\delta_kA_k\,\tfrac m2\|y_k-z_k\|^2 ,
$$

which is the reformulation of the proof of Theorem 4.3 with $1+qA_k$ replaced by $qA_k$. The first coefficient is nonnegative if and only if $a_k\delta_k\leq A_{k+1}$, that is, $a_k^2\leq qA_{k+1}^2$. $\square$

### 1.2 SymPy verification

To verify the expression symbolically, one can define the following symbols:

In [18]:
L = sp.symbols('L')
m = sp.symbols('m')
Ak = sp.symbols('A_k')
ak = sp.symbols('a_k')

xk = sp.symbols('x_k')
zk = sp.symbols('z_k')
yk = sp.symbols('y_k')
gyk = sp.symbols('\\nabla{}f(y_k)')
fyk = sp.symbols('f(y_k)')
fxk = sp.symbols('f(x_k)')
fxkp1 = sp.symbols('f(x_{k+1})')
Ark_zk = sp.symbols('A_k{}r_k(z_k)')

q = m / L
Akp1 = Ak + ak
deltak = ak / (q * Akp1)
zkp1 = zk + q * deltak * (yk - zk) - deltak / L * gyk

ellk_zkp1 = fyk + gyk * (zkp1 - yk) + m / 2 * (zkp1 - yk) ** 2
Ark_zkp1 = Ark_zk + m * Ak / 2 * (zkp1 - zk) ** 2            # quadratic, curvature m A_k, minimized at z_k
Arkp1_zkp1 = Ark_zkp1 + ak * ellk_zkp1

Now, as before, one can write the inequalities used in the proof (in the form "$\ldots\leqslant 0$") and their weights to form the weighted sum:

In [19]:
inequality1 = fyk - fxk + gyk * (xk - yk)
inequality2 = fxkp1 - fyk + 1 / 2 / L * gyk ** 2
inequality3 = gyk * (Ak * (yk - xk) + q * deltak * Ak * (yk - zk))
lambda1 = Ak
lambda2 = Akp1
lambda3 = 1

weighted_sum = lambda1 * inequality1 + lambda2 * inequality2 + lambda3 * inequality3

Similarly, one can define the two values of the gap and form the target inequality

In [20]:
potentialk = Ak * fxk - Ark_zk
potentialkp1 = Akp1 * fxkp1 - Arkp1_zkp1

target_inequality = potentialkp1 - potentialk + (Akp1 - ak * deltak) / 2 / L * gyk ** 2 + q * deltak * Ak * m / 2 * (yk - zk) ** 2

It remains to check that the expressions match, which we check by verifying their difference is identically zero

In [21]:
check(weighted_sum - target_inequality)

0

As complements: $z_{k+1}$ minimizes $A_{k+1}r_{k+1}$; the update of $z_{k+1}$ is the averaging $z_{k+1}=(1-\tau_k)z_k+\tau_k\big(y_k-\tfrac1m\nabla f(y_k)\big)$ with $\tau_k=q\delta_k=a_k/A_{k+1}$; the coefficient of the gradient term equals $\tfrac{qA_{k+1}^2-a_k^2}{2mA_{k+1}}$; the minimal value of the averaged model is the one displayed in the manuscript,
$$
v_{k+1}=(1-\tau_k)\,v_k+\tau_k\Big(f(y_k)-\tfrac1{2m}\|\nabla f(y_k)\|^2\Big)+\tfrac m2\tau_k(1-\tau_k)\Big\|z_k-y_k+\tfrac1m\nabla f(y_k)\Big\|^2,
$$
where $v_k=r_k(z_k)$ and $v_{k+1}=r_{k+1}(z_{k+1})$; and the largest admissible increment $a_k=\sqrt q\,A_{k+1}$, i.e. $A_{k+1}=A_k/(1-\sqrt q)$, gives the constant parameters $\tau_k=\sqrt q$ of the method (2.30): $y_k=x_k+\tfrac{\sqrt q}{1+\sqrt q}(z_k-x_k)$ and $z_{k+1}=(1-\sqrt q)z_k+\sqrt q\,(y_k-\tfrac1m\nabla f(y_k))$.

In [22]:
z = sp.symbols('z')
Arkp1_z = Ark_zk + m * Ak / 2 * (z - zk) ** 2 + ak * (fyk + gyk * (z - yk) + m / 2 * (z - yk) ** 2)
tauk = ak / Akp1
vk = Ark_zk / Ak
vkp1_claimed = (1 - tauk) * vk + tauk * (fyk - gyk ** 2 / 2 / m) + m / 2 * tauk * (1 - tauk) * (zk - yk + gyk / m) ** 2

sq = sp.sqrt(q)
Akp1_const = Ak / (1 - sq)
ak_const = Akp1_const - Ak
tau_const = sp.simplify(ak_const / Akp1_const)
yk_nesterov = (xk + tau_const * zk) / (1 + tau_const)
zkp1_const = zkp1.subs(ak, ak_const)

(check(sp.diff(Arkp1_z, z).subs(z, zkp1)),
 check(zkp1 - ((1 - tauk) * zk + tauk * (yk - gyk / m))),
 check((Akp1 - ak * deltak) / 2 / L - (q * Akp1 ** 2 - ak ** 2) / 2 / m / Akp1),
 check(Arkp1_zkp1 / Akp1 - vkp1_claimed),
 check(tau_const - sq),
 check(yk_nesterov - (xk + sq / (1 + sq) * (zk - xk))),
 check(zkp1_const - ((1 - sq) * zk + sq * (yk - gyk / m))))

(0, 0, 0, 0, 0, 0, 0)

## 2 Verification of Theorem 4.3

### 2.1 Statement and proof (as in the manuscript)

**Theorem 4.3. (Nesterov's method from models, $m=0$)** Let $f\in\mathcal{F}_{0,L}(\mathbb{R}^d)$, let $k\geq0$, $A_k,a_k\geq0$, let
$$
A_k\,r_k(x)=\tfrac L2\|x-x_0\|^2+\sum_{i=0}^{k-1}a_i\,\ell_i(x),\qquad \ell_i(x)=f(y_i)+\langle\nabla f(y_i),\,x-y_i\rangle,
$$
for some $y_0,\ldots,y_{k-1}\in\mathbb{R}^d$, with minimizer $z_k$, let $x_k,y_k\in\mathbb{R}^d$ satisfy the relation
$$
\langle\nabla f(y_k),\,A_kx_k+a_kz_k-A_{k+1}y_k\rangle\geq0,
$$
let $x_{k+1}=y_k-\tfrac1L\nabla f(y_k)$, $A_{k+1}r_{k+1}=A_kr_k+a_k\ell_k$ with minimizer $z_{k+1}$, and let $\Psi_k=A_k\big(f(x_k)-r_k(z_k)\big)$. If $a_k^2-a_k-A_k\leq0$, then it holds that
$$
\Psi_{k+1}\leqslant\Psi_k
$$
with $A_{k+1}=A_k+a_k$.

*Proof.* The proof consists in a weighted sum of the following inequalities, which are three of the four inequalities of the proof of Theorem 2.5:

- convexity between $x_k$ and $y_k$, with weight $\lambda_1=A_k$:

  $$
  f(x_k)\geq f(y_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle,
  $$

- smoothness between $x_{k+1}$ and $y_k$ via the descent lemma, with weight $\lambda_2=A_{k+1}$:

  $$
  f(x_{k+1})\leq f(y_k)-\tfrac{1}{2L}\|\nabla f(y_k)\|^2,
  $$

- the relation, with weight $\lambda_3=1$:

  $$
  \langle\nabla f(y_k),\,A_kx_k+a_kz_k-A_{k+1}y_k\rangle\geq0.
  $$

The weighted sum reads

$$
\begin{aligned}
0\geq{}&\lambda_1\big[f(y_k)-f(x_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle\big]\\
&+\lambda_2\big[f(x_{k+1})-f(y_k)+\tfrac{1}{2L}\|\nabla f(y_k)\|^2\big]\\
&+\lambda_3\langle\nabla f(y_k),\,A_{k+1}y_k-A_kx_k-a_kz_k\rangle ,
\end{aligned}
$$

which, using the update $z_{k+1}=z_k-\tfrac{a_k}{L}\nabla f(y_k)$ and $A_{k+1}=A_k+a_k$, together with $A_{k+1}r_{k+1}(z_{k+1})=A_kr_k(z_{k+1})+a_k\ell_k(z_{k+1})$ and $A_kr_k(z_{k+1})=A_kr_k(z_k)+\tfrac L2\|z_{k+1}-z_k\|^2$ (as $A_kr_k$ is a quadratic with curvature $L$ minimized at $z_k$), can be reformulated exactly as

$$
0\geq\Psi_{k+1}-\Psi_k+\tfrac{A_{k+1}-a_k^2}{2L}\|\nabla f(y_k)\|^2 ,
$$

which is the claim as soon as $A_{k+1}\geq a_k^2$. $\square$

### 2.2 SymPy verification

To verify the expression symbolically, one can define the following symbols. The minimal value $A_kr_k(z_k)$ of the current model is the free symbol `Ark_zk`, and the model is evaluated at $z_{k+1}$ through its quadratic expansion around $z_k$:

In [2]:
L = sp.symbols('L')
Ak = sp.symbols('A_k')
ak = sp.symbols('a_k')

xk = sp.symbols('x_k')
zk = sp.symbols('z_k')
yk = sp.symbols('y_k')
gyk = sp.symbols('\\nabla{}f(y_k)')
fyk = sp.symbols('f(y_k)')
fxk = sp.symbols('f(x_k)')
fxkp1 = sp.symbols('f(x_{k+1})')
Ark_zk = sp.symbols('A_k{}r_k(z_k)')

Akp1 = Ak + ak
zkp1 = zk - ak / L * gyk

ellk_zkp1 = fyk + gyk * (zkp1 - yk)           # ell_k(z_{k+1})
Ark_zkp1 = Ark_zk + L / 2 * (zkp1 - zk) ** 2  # A_k r_k(z_{k+1})
Arkp1_zkp1 = Ark_zkp1 + ak * ellk_zkp1        # A_{k+1} r_{k+1}(z_{k+1}) = A_k r_k(z_{k+1}) + a_k ell_k(z_{k+1})

Now, one can write the three inequalities used in the proof (in the form "$\ldots\leqslant 0$") and their weights to form the weighted sum:

In [3]:
inequality1 = fyk - fxk + gyk * (xk - yk)
inequality2 = fxkp1 - fyk + 1 / 2 / L * gyk ** 2
inequality3 = - gyk * (Ak * xk + ak * zk - Akp1 * yk)
lambda1 = Ak
lambda2 = Akp1
lambda3 = 1

weighted_sum = lambda1 * inequality1 + lambda2 * inequality2 + lambda3 * inequality3

Similarly, one can define the two values of the gap and form the target inequality

In [4]:
potentialk = Ak * fxk - Ark_zk
potentialkp1 = Akp1 * fxkp1 - Arkp1_zkp1

target_inequality = potentialkp1 - potentialk + (Akp1 - ak ** 2) / 2 / L * gyk ** 2

It remains to check that the expressions match, which we check by verifying their difference is identically zero

In [5]:
check(weighted_sum - target_inequality)

0

## 3 Verification of Theorem 4.5

### 3.1 Statement and proof (as in the manuscript)

**Theorem 4.5. (Nesterov's method from models, $m\geq0$)** Let $f\in\mathcal{F}_{m,L}(\mathbb{R}^d)$ with $0\leq m<L$, $q=m/L$, let $k\geq0$, $A_k,a_k\geq0$, $\delta_k=\tfrac{a_k}{1+qA_{k+1}}$, let
$$
A_k\,r_k(x)=\tfrac L2\|x-x_0\|^2+\sum_{i=0}^{k-1}a_i\,\ell_i(x),\qquad \ell_i(x)=f(y_i)+\langle\nabla f(y_i),\,x-y_i\rangle+\tfrac m2\|x-y_i\|^2,
$$
for some $y_0,\ldots,y_{k-1}\in\mathbb{R}^d$, with minimizer $z_k$, let $x_k,y_k\in\mathbb{R}^d$ satisfy the relation
$$
\langle\nabla f(y_k),\,A_k(x_k-y_k)+\delta_k(1+qA_k)(z_k-y_k)\rangle\geq0,
$$
let $x_{k+1}=y_k-\tfrac1L\nabla f(y_k)$, $A_{k+1}r_{k+1}=A_kr_k+a_k\ell_k$ with minimizer
$$
z_{k+1}=z_k+q\delta_k\,(y_k-z_k)-\tfrac{\delta_k}{L}\nabla f(y_k),
$$
and let $\Psi_k=A_k\big(f(x_k)-r_k(z_k)\big)$. If $A_{k+1}(1+qA_{k+1})-a_k^2\geq0$, then it holds that
$$
\Psi_{k+1}\leqslant\Psi_k
$$
with $A_{k+1}=A_k+a_k$.

*Proof.* The proof consists in a weighted sum of the three inequalities of the proof of Theorem 4.1, the relation being adapted to carry the strongly convex component:

- convexity between $x_k$ and $y_k$, with weight $\lambda_1=A_k$:

  $$
  f(x_k)\geq f(y_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle,
  $$

- the descent lemma, with weight $\lambda_2=A_{k+1}$:

  $$
  f(x_{k+1})\leq f(y_k)-\tfrac{1}{2L}\|\nabla f(y_k)\|^2,
  $$

- the relation, with weight $\lambda_3=1$:

  $$
  \langle\nabla f(y_k),\,A_k(x_k-y_k)+\delta_k(1+qA_k)(z_k-y_k)\rangle\geq0.
  $$

The weighted sum reads

$$
\begin{aligned}
0\geq{}&\lambda_1\big[f(y_k)-f(x_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle\big]\\
&+\lambda_2\big[f(x_{k+1})-f(y_k)+\tfrac{1}{2L}\|\nabla f(y_k)\|^2\big]\\
&+\lambda_3\langle\nabla f(y_k),\,A_k(y_k-x_k)+\delta_k(1+qA_k)(y_k-z_k)\rangle ,
\end{aligned}
$$

which, using the update of $z_{k+1}$ and $A_{k+1}=A_k+a_k$, together with $A_{k+1}r_{k+1}(z_{k+1})=A_kr_k(z_{k+1})+a_k\ell_k(z_{k+1})$ and $A_kr_k(z_{k+1})=A_kr_k(z_k)+\tfrac{L+mA_k}{2}\|z_{k+1}-z_k\|^2$ (as $A_kr_k$ is a quadratic with curvature $L+mA_k$ minimized at $z_k$), can be reformulated exactly as

$$
0\geq\Psi_{k+1}-\Psi_k+\tfrac{A_{k+1}-a_k\delta_k}{2L}\|\nabla f(y_k)\|^2+\delta_k(1+qA_k)\,\tfrac m2\|y_k-z_k\|^2 ,
$$

which is the reformulation of the proof of Theorem 2.11, with $\Psi_k$ in place of $\Phi_k$. Both coefficients are nonnegative under our assumption. $\square$

### 3.2 SymPy verification

To verify the expression symbolically, one can define the following symbols:

In [8]:
L = sp.symbols('L')
m = sp.symbols('m')
Ak = sp.symbols('A_k')
ak = sp.symbols('a_k')

xk = sp.symbols('x_k')
zk = sp.symbols('z_k')
yk = sp.symbols('y_k')
gyk = sp.symbols('\\nabla{}f(y_k)')
fyk = sp.symbols('f(y_k)')
fxk = sp.symbols('f(x_k)')
fxkp1 = sp.symbols('f(x_{k+1})')
Ark_zk = sp.symbols('A_k{}r_k(z_k)')

q = m / L
Akp1 = Ak + ak
deltak = ak / (1 + q * Akp1)
zkp1 = zk + q * deltak * (yk - zk) - deltak / L * gyk

ellk_zkp1 = fyk + gyk * (zkp1 - yk) + m / 2 * (zkp1 - yk) ** 2
Ark_zkp1 = Ark_zk + (L + m * Ak) / 2 * (zkp1 - zk) ** 2      # quadratic, curvature L + m A_k, minimized at z_k
Arkp1_zkp1 = Ark_zkp1 + ak * ellk_zkp1

Now, as before, one can write the inequalities used in the proof (in the form "$\ldots\leqslant 0$") and their weights to form the weighted sum:

In [9]:
inequality1 = fyk - fxk + gyk * (xk - yk)
inequality2 = fxkp1 - fyk + 1 / 2 / L * gyk ** 2
inequality3 = gyk * (Ak * (yk - xk) + deltak * (1 + q * Ak) * (yk - zk))
lambda1 = Ak
lambda2 = Akp1
lambda3 = 1

weighted_sum = lambda1 * inequality1 + lambda2 * inequality2 + lambda3 * inequality3

Similarly, one can define the two values of the gap and form the target inequality

In [10]:
potentialk = Ak * fxk - Ark_zk
potentialkp1 = Akp1 * fxkp1 - Arkp1_zkp1

target_inequality = potentialkp1 - potentialk + (Akp1 - ak * deltak) / 2 / L * gyk ** 2 + deltak * (1 + q * Ak) * m / 2 * (yk - zk) ** 2

It remains to check that the expressions match, which we check by verifying their difference is identically zero

In [11]:
check(weighted_sum - target_inequality)

0